In [1]:
import json
from pathlib import Path

import pandas as pd
from pandas import json_normalize

RAW = Path("..") / "data" / "raw"
print(Path.cwd())
print((RAW / "events_nested.jsonl").exists(), (RAW / "alerts_by_file.json").exists())

d:\DEV\learn\M3\notebooks
True True


Đọc JSON Lines, mỗi dòng là một event

In [ ]:
nested = pd.read_json(RAW / "events_nested.jsonl",lines=True)
print(nested.shape)
print(nested.columns.tolist())
nested.loc[0,"winlog"] # lấy nhãn index 0 và cột có tên "winlog"

(4633, 5)
['@timestamp', 'event', 'winlog', 'labels', 'tags']


{'channel': 'Security',
 'computer_name': 'PC02.example.corp',
 'record_id': 5278,
 'event_data': {'ProcessName': 'C:\\Windows\\System32\\services.exe',
  'SubjectUserName': 'PC02$',
  'TargetUserName': 'SYSTEM',
  'LogonType': 5}}

Làm phẳng toàn bộ
to_dict chuyển DataFrame thành dict / list của Python
"records" nghĩa là “mỗi dòng là một bản ghi”

In [3]:
flat = json_normalize(nested.to_dict("records"),sep ="_")
print(flat.columns.tolist())

['@timestamp', 'tags', 'event_code', 'winlog_channel', 'winlog_computer_name', 'winlog_record_id', 'winlog_event_data_ProcessName', 'winlog_event_data_SubjectUserName', 'winlog_event_data_TargetUserName', 'winlog_event_data_LogonType', 'labels_evtx_file', 'labels_tactic', 'winlog_event_data_IpAddress', 'winlog_event_data_Application', 'winlog_event_data_NewProcessName', 'winlog_event_data_Image', 'winlog_event_data_User', 'winlog_event_data_DestinationIp', 'winlog_event_data_DestinationPort', 'winlog_event_data_ParentImage', 'winlog_event_data_CommandLine', 'winlog_event_data_ParentCommandLine', 'winlog_event_data_TargetFilename']


Chỉ làm phẳng 1 tầng

In [4]:
flat1 = json_normalize(nested.to_dict("records"), max_level=1, sep="_").columns.tolist()
print(flat1)

['@timestamp', 'tags', 'event_code', 'winlog_channel', 'winlog_computer_name', 'winlog_record_id', 'winlog_event_data', 'labels_evtx_file', 'labels_tactic']


Đếm event_code sau khi làm phẳng

In [5]:
flat["event_code"].value_counts().head(3)

event_code
5145    950
1       843
7       613
Name: count, dtype: int64

Phần C File thật: alerts_by_file.json
Mỗi file EVTX là một alert chứa danh sách events

In [8]:
alerts = json.loads((RAW / "alerts_by_file.json").read_text(encoding="utf-8"))
print(len(alerts))
alerts[0]

249


{'alert_id': '4624 LT3 AnonymousLogon Localhost - JuicyPotato.evtx',
 'tactic': 'Privilege Escalation',
 'host': {'name': 'alice.insecurebank.local'},
 'event_count': 3,
 'events': [{'EventID': 1102,
   'Channel': 'Security',
   'SystemTime': '2019-11-15 08:19:02.298512'},
  {'EventID': 4624,
   'Channel': 'Security',
   'SystemTime': '2019-11-15 08:19:16.102510'},
  {'EventID': 4634,
   'Channel': 'Security',
   'SystemTime': '2019-11-15 08:19:17.134468'}]}

Bung events, giữ thông tin của alert cha

In [ ]:
ev_alerts = json_normalize(alerts, record_path="events",
                          meta=["alert_id", "tactic", ["host", "name"]], sep="_")
print(ev_alerts.shape)   # (4633, 9)
ev_alerts.head()

(4633, 9)


,EventID,Channel,SystemTime,Image,CommandLine,ParentImage,alert_id,tactic,host_name
0,1102,Security,2019-11-15 08:19:02.298512,NaN,NaN,NaN,4624 LT3 AnonymousLogon Localhost - JuicyPotat...,Privilege Escalation,alice.insecurebank.local
1,4624,Security,2019-11-15 08:19:16.102510,NaN,NaN,NaN,4624 LT3 AnonymousLogon Localhost - JuicyPotat...,Privilege Escalation,alice.insecurebank.local
2,4634,Security,2019-11-15 08:19:17.134468,NaN,NaN,NaN,4624 LT3 AnonymousLogon Localhost - JuicyPotat...,Privilege Escalation,alice.insecurebank.local
3,4661,Security,2017-06-12 23:39:43.512987,NaN,NaN,NaN,4765_sidhistory_add_t1178.evtx,Privilege Escalation,2012r2srv.maincorp.local
4,4765,Security,2017-06-12 23:39:43.512987,NaN,NaN,NaN,4765_sidhistory_add_t1178.evtx,Privilege Escalation,2012r2srv.maincorp.local


Đếm alert theo tactic và kiểm tra event_count và sắp giảm dần

In [11]:
print(ev_alerts.groupby("tactic")["alert_id"].nunique().sort_values(ascending=False))

al = json_normalize(alerts,sep="_")
(al.set_index("alert_id")["event_count"] == ev_alerts.groupby("alert_id").size()).all()

tactic
Privilege Escalation    61
Lateral Movement        47
Defense Evasion         36
Credential Access       35
Execution               33
Persistence             22
Discovery               10
Command and Control      5
Name: alert_id, dtype: int64


np.True_

Phần B — Làm sạch evtx_data.csv

In [12]:
# Đọc dữ liệu và chọn cột
df = pd.read_csv(RAW / "evtx_data.csv",low_memory=False)
df = df.drop(columns=["Unnamed: 0"])

COLS = [
    "EVTX_FileName", "EVTX_Tactic", "Channel", "EventID", "EventRecordID",
    "SystemTime", "Computer", "Image", "ParentImage", "CommandLine",
    "NewProcessName", "ProcessName", "Application",
]
ev = df[COLS].copy()
print(ev.shape)

(4633, 13)


In [13]:
# Vì sao to_datetime báo lỗi
print(ev["SystemTime"].head(3).tolist())
print(ev["SystemTime"].dtype)   # str

try:
    pd.to_datetime(ev["SystemTime"])
except ValueError as e:
    print("Lỗi:", str(e)[:120])
# Lỗi: time data "2019-11-26 23:55:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f" ...

no_frac = ~ev["SystemTime"].str.contains(".", regex=False)
print(no_frac.sum())   # 355
ev.loc[no_frac, "SystemTime"].head(2).tolist()   # ['2019-11-26 23:55:00', ...]

['2019-02-13 15:14:52.409735', '2019-02-13 15:15:04.175283', '2019-02-13 15:15:04.635946']
str
Lỗi: time data "2019-11-26 23:55:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f". You might want to try:
    - passing `forma
355


['2019-11-26 23:55:00', '2019-11-26 23:55:00']

In [14]:
# Chuyển sang kiểu thời gian đúng cách
ev["ts"] = pd.to_datetime(ev["SystemTime"], format="mixed", errors="coerce", utc=True)

print(ev["ts"].dtype)        # datetime64[us, UTC]
print(ev["ts"].isna().sum()) # 0
print(ev["ts"].min(), ev["ts"].max())
# 2017-06-09 19:21:26.968670+00:00  2021-06-09 05:18:11.103153+00:00
ev["ts"].dt.year.value_counts().sort_index()
# 2017: 4, 2018: 48, 2019: 3232, 2020: 1284, 2021: 65


datetime64[us, UTC]
0
2017-06-09 19:21:26.968670+00:00 2021-06-09 05:18:11.103153+00:00


ts
2017       4
2018      48
2019    3232
2020    1284
2021      65
Name: count, dtype: int64

In [15]:
# Dùng .dt và tính khoảng thời gian
ev["ts"].dt.hour.value_counts().head(3)   # 14h: 910, 11h: 671, 9h: 433

span = ev.groupby("EVTX_FileName")["ts"].agg(["min", "max"])
span["duration"] = span["max"] - span["min"]
print(span["duration"].median())   # 0 days 00:00:07.841277
span.sort_values("duration", ascending=False).head(3)
# rogue_msi_url_1040_1042.evtx  -> 226 days ...


0 days 00:00:07.841277


,min,max,duration
EVTX_FileName,,,
rogue_msi_url_1040_1042.evtx,2019-03-19 13:05:42+00:00,2019-10-31 14:48:42+00:00,226 days 01:43:00
DE_RDP_Tunneling_TerminalServices-RemoteConnectionManagerOperational_1149.evtx,2018-11-06 21:31:54.070986+00:00,2019-02-13 18:04:57.452387+00:00,98 days 20:33:03.381401
Exec_via_cpl_Application_Experience_EventID_17_ControlPanelApplet.evtx,2019-07-18 21:50:20.049400+00:00,2019-08-23 11:17:00.483376+00:00,35 days 13:26:40.433976


In [16]:
# Thông tin tiến trình nằm ở đâu
PROC_COLS = ["Image", "NewProcessName", "ProcessName", "Application"]
print(ev[PROC_COLS].notna().sum())
# Image 2140, NewProcessName 35, ProcessName 220, Application 106

ev[PROC_COLS].notna().sum(axis=1).value_counts()
# 1    2501
# 0    2132

Image             2140
NewProcessName      35
ProcessName        220
Application        106
dtype: int64


1    2501
0    2132
Name: count, dtype: int64

In [ ]:
# Tạo process_path và process_name
proc = (ev["Image"]
        .fillna(ev["NewProcessName"])
        .fillna(ev["ProcessName"])
        .fillna(ev["Application"]))
print(proc.notna().sum(), (proc == "-").sum())   # 2501 43

ev["process_path"] = proc.replace({"-": pd.NA, "": pd.NA})
ev["process_name"] = (ev["process_path"]
                      .str.extract(r"([^\\/]+)$", expand=False)
                      .str.lower())

2501 43


In [18]:
# Kiểm tra kết quả chuẩn hóa
print(round(ev["process_name"].notna().mean(), 3))   # 0.531
print(ev["process_name"].nunique())                   # 196

(ev.loc[ev["process_path"].notna(), ["process_path", "process_name"]]
   .drop_duplicates("process_name")
   .head(4))
# C:\Windows\System32\services.exe                      services.exe
# \device\harddiskvolume1\windows\system32\svchost.exe  svchost.exe

ev["process_name"].value_counts().head(5)
# svchost.exe 353, consent.exe 256, cmd.exe 131, system 103, lsass.exe 98

0.531
196


process_name
svchost.exe    353
consent.exe    256
cmd.exe        131
system         103
lsass.exe       98
Name: count, dtype: int64

In [22]:
# Dữ liệu thiếu
ev.isna().sum().sort_values(ascending=False).head(6)

print(ev.dropna().shape) # xóa mọi dòng có ít nhất một ô thiếu, và kết quả còn 0 dòng. Đây là lý do không bao giờ gọi dropna() trống trên bảng log thưa. mặc đinh xóa dòng ko xóa cột
print(ev.dropna(subset=["process_name"]).shape) # chỉ xóa dòng thiếu ở cột này
print(ev.loc[ev["process_name"].isna(), "Channel"].value_counts().head(3))
ev["process_name"].fillna("None").value_counts().head(3)

(0, 16)
(2458, 16)
Channel
Security                                1178
Application                              393
Microsoft-Windows-Sysmon/Operational     261
Name: count, dtype: int64


process_name
None           2175
svchost.exe     353
consent.exe     256
Name: count, dtype: int64

In [23]:
# Dữ liệu trùng
print(df.duplicated().sum())   # 0
print(ev.duplicated().sum())   # 0

no_rec = [c for c in ev.columns if c != "EventRecordID"]
print(ev.duplicated(subset=no_rec).sum())   # 1017

mask = ev.duplicated(subset=no_rec, keep=False)
(ev.loc[mask, ["EVTX_FileName", "EventID", "EventRecordID", "SystemTime"]]
   .sort_values(["EVTX_FileName", "SystemTime"])
   .head(3))
# ACL_ForcePwd_...evtx  5136  198238043  2019-03-25 10:33:56.457630
# ACL_ForcePwd_...evtx  5136  198238044  2019-03-25 10:33:56.457630

KEY = ["EVTX_FileName", "Channel", "EventRecordID"]
ev.duplicated(subset=KEY).sum()   # 0

0
0
1017


np.int64(0)

In [26]:
# Kiểu category
print(ev["EVTX_Tactic"].memory_usage(deep=True), ev["Channel"].memory_usage(deep=True)) #tính bằng byte
ev["EVTX_Tactic"] = ev["EVTX_Tactic"].astype("category") # lưu mỗi giá trị khác nhau một lần (8 tactic, 12 kênh), còn mỗi dòng chỉ lưu một mã số nhỏ trỏ tới giá trị đó. Giống một enum.
ev["Channel"] = ev["Channel"].astype("category")

print(ev["EVTX_Tactic"].memory_usage(deep=True), ev["Channel"].memory_usage(deep=True))
ev["EVTX_Tactic"].cat.categories.tolist() # xem danh sách giá trị của category.


5273 5836
5273 5836


['Command and Control',
 'Credential Access',
 'Defense Evasion',
 'Discovery',
 'Execution',
 'Lateral Movement',
 'Persistence',
 'Privilege Escalation']

In [27]:
# Kiểm tra schema cuối cùng
assert ev["ts"].isna().sum() == 0, "có thời gian không đọc được" #assert điều_kiện, "thông báo"
assert not ev.duplicated(subset=KEY).any(), "có bản ghi trùng"

print(ev.shape)                                           # (4633, 16)
print(round(ev.memory_usage(deep=True).sum() / 1e6, 2))   # 2.78 (MB)
ev.dtypes
# EVTX_Tactic category, Channel category, EventID int64, ts datetime64[us, UTC],
# process_path str, process_name str, ...

(4633, 16)
2.78


EVTX_FileName                     str
EVTX_Tactic                  category
Channel                      category
EventID                         int64
EventRecordID                   int64
SystemTime                        str
Computer                          str
Image                             str
ParentImage                       str
CommandLine                       str
NewProcessName                    str
ProcessName                       str
Application                       str
ts                datetime64[us, UTC]
process_path                      str
process_name                      str
dtype: object